In [ ]:
import pandas as pd

df_2022 = pd.read_csv("bdqueimadas_2022-01-01_2022-12-31.csv")
df_2023 = pd.read_csv("bdqueimadas_2023-01-01_2023-12-31.csv")
df_2024 = pd.read_csv("bdqueimadas_2024-01-01_2024-12-31.csv")

for nombre, d in [("2022", df_2022), ("2023", df_2023), ("2024", df_2024)]:
    print(f"--- {nombre} ---")
    print("Shape:", d.shape)
    print("Columnas:", d.columns.tolist())
    print()

--- 2022 ---
Shape: (115033, 12)
Columnas: ['DataHora', 'Satelite', 'Pais', 'Estado', 'Municipio', 'Bioma', 'DiaSemChuva', 'Precipitacao', 'RiscoFogo', 'FRP', 'Latitude', 'Longitude']

--- 2023 ---
Shape: (98639, 12)
Columnas: ['DataHora', 'Satelite', 'Pais', 'Estado', 'Municipio', 'Bioma', 'DiaSemChuva', 'Precipitacao', 'RiscoFogo', 'FRP', 'Latitude', 'Longitude']

--- 2024 ---
Shape: (140346, 12)
Columnas: ['DataHora', 'Satelite', 'Pais', 'Estado', 'Municipio', 'Bioma', 'DiaSemChuva', 'Precipitacao', 'RiscoFogo', 'FRP', 'Latitude', 'Longitude']



In [ ]:
df_queimadas = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

print("Total combinado:", df_queimadas.shape)
print(df_queimadas.dtypes)
print()
df_queimadas.head()

Total combinado: (354018, 12)
DataHora         object
Satelite         object
Pais             object
Estado           object
Municipio        object
Bioma            object
DiaSemChuva       int64
Precipitacao    float64
RiscoFogo       float64
FRP             float64
Latitude        float64
Longitude       float64
dtype: object



,DataHora,Satelite,Pais,Estado,Municipio,Bioma,DiaSemChuva,Precipitacao,RiscoFogo,FRP,Latitude,Longitude
0,2022/01/01 18:13:00,AQUA_M-T,Brasil,RONDÔNIA,GUAJARÁ-MIRIM,Amazônia,8,0.2,0.2,11.8,-11.73219,-64.88873
1,2022/01/01 18:13:00,AQUA_M-T,Brasil,RONDÔNIA,GUAJARÁ-MIRIM,Amazônia,8,0.1,0.6,12.0,-11.72939,-64.88394
2,2022/01/01 18:13:00,AQUA_M-T,Brasil,ACRE,XAPURI,Amazônia,7,0.0,0.0,5.5,-10.97156,-68.33749
3,2022/01/01 18:15:00,AQUA_M-T,Brasil,RONDÔNIA,NOVA MAMORÉ,Amazônia,3,0.0,0.1,23.2,-10.14938,-65.11926
4,2022/01/01 18:15:00,AQUA_M-T,Brasil,AMAZONAS,BOCA DO ACRE,Amazônia,4,0.4,0.0,6.7,-9.05768,-67.50098


In [ ]:
print("Valores nulos por columna:")
print(df_queimadas.isnull().sum())
print()
print("Rango de fechas:")
print(df_queimadas['DataHora'].min(), "→", df_queimadas['DataHora'].max())
print()
print("Biomas presentes:", df_queimadas['Bioma'].unique())
print("Satélites presentes:", df_queimadas['Satelite'].unique())

Valores nulos por columna:
DataHora        0
Satelite        0
Pais            0
Estado          0
Municipio       0
Bioma           0
DiaSemChuva     0
Precipitacao    0
RiscoFogo       0
FRP             0
Latitude        0
Longitude       0
dtype: int64

Rango de fechas:
2022/01/01 18:13:00 → 2024/12/31 17:53:00

Biomas presentes: ['Amazônia']
Satélites presentes: ['AQUA_M-T']


In [ ]:
df_queimadas["DataHora"] = pd.to_datetime(df_queimadas["DataHora"], format="%Y/%m/%d %H:%M:%S", errors="coerce")

print("Fechas inválidas tras conversión:", df_queimadas["DataHora"].isna().sum())

# separamos fecha (para cruzar con DETER) y hora (por si sirve para análisis de patrones diarios)
df_queimadas["fecha"] = df_queimadas["DataHora"].dt.date
df_queimadas["fecha"] = pd.to_datetime(df_queimadas["fecha"])
df_queimadas["hora"] = df_queimadas["DataHora"].dt.hour

print(df_queimadas[["DataHora", "fecha", "hora"]].head())

Fechas inválidas tras conversión: 0
             DataHora      fecha  hora
0 2022-01-01 18:13:00 2022-01-01    18
1 2022-01-01 18:13:00 2022-01-01    18
2 2022-01-01 18:13:00 2022-01-01    18
3 2022-01-01 18:15:00 2022-01-01    18
4 2022-01-01 18:15:00 2022-01-01    18


In [ ]:
print(df_queimadas[["DiaSemChuva", "Precipitacao", "RiscoFogo", "FRP"]].describe())

         DiaSemChuva   Precipitacao      RiscoFogo            FRP
count  354018.000000  354018.000000  354018.000000  354018.000000
mean       -1.266097       1.025759      -2.858251      60.632525
std       129.653816       4.323705      58.700080     142.159092
min      -999.000000       0.000000    -999.000000       0.000000
25%         2.000000       0.000000       0.220000      14.300000
50%         5.000000       0.000000       0.690000      26.600000
75%        14.000000       0.000000       1.000000      55.600000
max       336.000000     119.370000       1.000000    8094.300000


In [ ]:
import numpy as np

# antes de reemplazar, contemos cuántos hay en cada columna
print("Filas con -999 en DiaSemChuva:", (df_queimadas["DiaSemChuva"] == -999).sum())
print("Filas con -999 en RiscoFogo:", (df_queimadas["RiscoFogo"] == -999).sum())

Filas con -999 en DiaSemChuva: 5666
Filas con -999 en RiscoFogo: 1225


In [ ]:
df_queimadas["DiaSemChuva"] = df_queimadas["DiaSemChuva"].replace(-999, np.nan)
df_queimadas["RiscoFogo"] = df_queimadas["RiscoFogo"].replace(-999, np.nan)

print("\nDescribe después de limpiar -999:")
print(df_queimadas[["DiaSemChuva", "Precipitacao", "RiscoFogo", "FRP"]].describe())


Describe después de limpiar -999:
         DiaSemChuva   Precipitacao      RiscoFogo            FRP
count  348352.000000  354018.000000  352793.000000  354018.000000
mean       14.962202       1.025759       0.600643      60.632525
std        25.071961       4.323705       0.375864     142.159092
min         0.000000       0.000000       0.000000       0.000000
25%         2.000000       0.000000       0.220000      14.300000
50%         5.000000       0.000000       0.690000      26.600000
75%        14.000000       0.000000       1.000000      55.600000
max       336.000000     119.370000       1.000000    8094.300000


In [ ]:
print("Municipios únicos en BDQueimadas:", df_queimadas["Municipio"].nunique())
print(df_queimadas["Municipio"].sort_values().unique()[:20])  # primeros 20, para ver el formato

Municipios únicos en BDQueimadas: 538
['ABAETETUBA' 'ABEL FIGUEIREDO' 'ACARÁ' 'ACRELÂNDIA' 'AFUÁ' 'ALCÂNTARA'
 'ALENQUER' 'ALMEIRIM' 'ALTA FLORESTA' "ALTA FLORESTA D'OESTE" 'ALTAMIRA'
 'ALTAMIRA DO MARANHÃO' 'ALTO ALEGRE' 'ALTO ALEGRE DO PINDARÉ'
 'ALTO ALEGRE DOS PARECIS' 'ALTO BOA VISTA' 'ALTO PARAGUAI' 'ALTO PARAÍSO'
 'ALVARÃES' "ALVORADA D'OESTE"]


In [ ]:
import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto

df_queimadas["municipio_norm"] = df_queimadas["Municipio"].apply(normalizar_texto)

print("Municipios únicos en BDQueimadas:", df_queimadas["municipio_norm"].nunique())
print(df_queimadas["municipio_norm"].sort_values().unique()[:20])

Municipios únicos en BDQueimadas: 538
['ABAETETUBA' 'ABEL FIGUEIREDO' 'ACAILANDIA' 'ACARA' 'ACRELANDIA' 'AFUA'
 'AGUA AZUL DO NORTE' 'ALCANTARA' 'ALENQUER' 'ALMEIRIM' 'ALTA FLORESTA'
 "ALTA FLORESTA D'OESTE" 'ALTAMIRA' 'ALTAMIRA DO MARANHAO' 'ALTO ALEGRE'
 'ALTO ALEGRE DO PINDARE' 'ALTO ALEGRE DOS PARECIS' 'ALTO BOA VISTA'
 'ALTO PARAGUAI' 'ALTO PARAISO']


In [ ]:
df_deter = pd.read_csv("deter_limpio.csv")

municipios_deter = set(df_deter["municipio_norm"].unique())
municipios_queimadas = set(df_queimadas["municipio_norm"].unique())

print(f"Municipios en DETER: {len(municipios_deter)}")
print(f"Municipios en BDQueimadas: {len(municipios_queimadas)}")
print(f"Municipios en común: {len(municipios_deter & municipios_queimadas)}")
print(f"En DETER pero NO en BDQueimadas: {len(municipios_deter - municipios_queimadas)}")

Municipios en DETER: 424
Municipios en BDQueimadas: 538
Municipios en común: 417
En DETER pero NO en BDQueimadas: 7


In [ ]:
sin_match = municipios_deter - municipios_queimadas
print("Municipios de DETER sin match en BDQueimadas:")
for m in sorted(sin_match):
    print(f"  - {m}")

Municipios de DETER sin match en BDQueimadas:
  - BARAO DE MELGACO
  - CARMOLANDIA
  - ELDORADO DOS CARAJAS
  - NOSSA SENHORA DO LIVRAMENTO
  - NOVO SANTO ANTONIO
  - SANTA ISABEL DO PARA
  - SERRA NOVA DOURADA


In [ ]:
for m in sorted(sin_match):
    # busca nombres parecidos en BDQueimadas (primeras letras en común)
    parecidos = [x for x in municipios_queimadas if x.startswith(m[:5])]
    print(f"'{m}' → posibles coincidencias: {parecidos}")

'BARAO DE MELGACO' → posibles coincidencias: []
'CARMOLANDIA' → posibles coincidencias: []
'ELDORADO DOS CARAJAS' → posibles coincidencias: ['ELDORADO DO CARAJAS']
'NOSSA SENHORA DO LIVRAMENTO' → posibles coincidencias: []
'NOVO SANTO ANTONIO' → posibles coincidencias: ['NOVO PROGRESSO', 'NOVO HORIZONTE DO OESTE', 'NOVO ARIPUANA', 'NOVO HORIZONTE DO NORTE', 'NOVO MUNDO', 'NOVO AIRAO', 'NOVO REPARTIMENTO']
'SANTA ISABEL DO PARA' → posibles coincidencias: ['SANTA LUZIA DO PARUA', 'SANTA BARBARA DO PARA', 'SANTA ISABEL DO RIO NEGRO', 'SANTA CARMEM', 'SANTANA DO ARAGUAIA', 'SANTA MARIA DAS BARREIRAS', 'SANTA IZABEL DO PARA', 'SANTA TEREZINHA', 'SANTA LUZIA', 'SANTA HELENA', 'SANTA FE DO ARAGUAIA', 'SANTANA', 'SANTAREM NOVO', 'SANTA MARIA DO PARA', 'SANTA ROSA DO PURUS', "SANTA LUZIA D'OESTE", 'SANTA CRUZ DO ARARI', 'SANTA RITA', 'SANTA CRUZ DO XINGU', 'SANTA INES', 'SANTA LUZIA DO PARA', 'SANTAREM']
'SERRA NOVA DOURADA' → posibles coincidencias: ['SERRA DO NAVIO', 'SERRANO DO MARANHAO']


In [ ]:
correcciones_municipio = {
    "ELDORADO DOS CARAJAS": "ELDORADO DO CARAJAS",
    "SANTA ISABEL DO PARA": "SANTA IZABEL DO PARA",
}

df_deter["municipio_norm"] = df_deter["municipio_norm"].replace(correcciones_municipio)


df_deter.to_csv("deter_limpio_merge.csv", index=False, encoding="utf-8")
print("deter_limpio_merge.csv actualizado con correcciones de nomenclatura")

deter_limpio_merge.csv actualizado con correcciones de nomenclatura


In [ ]:
municipios_deter = set(df_deter["municipio_norm"].unique())
municipios_queimadas = set(df_queimadas["municipio_norm"].unique())

print(f"Municipios en común: {len(municipios_deter & municipios_queimadas)}")
print(f"En DETER pero NO en BDQueimadas: {len(municipios_deter - municipios_queimadas)}")
print(sorted(municipios_deter - municipios_queimadas))

Municipios en común: 419
En DETER pero NO en BDQueimadas: 5
['BARAO DE MELGACO', 'CARMOLANDIA', 'NOSSA SENHORA DO LIVRAMENTO', 'NOVO SANTO ANTONIO', 'SERRA NOVA DOURADA']


In [ ]:
df_queimadas.head()

,DataHora,Satelite,Pais,Estado,Municipio,Bioma,DiaSemChuva,Precipitacao,RiscoFogo,FRP,Latitude,Longitude,fecha,hora,municipio_norm
0,2022-01-01 18:13:00,AQUA_M-T,Brasil,RONDÔNIA,GUAJARÁ-MIRIM,Amazônia,8.0,0.2,0.2,11.8,-11.73219,-64.88873,2022-01-01,18,GUAJARA-MIRIM
1,2022-01-01 18:13:00,AQUA_M-T,Brasil,RONDÔNIA,GUAJARÁ-MIRIM,Amazônia,8.0,0.1,0.6,12.0,-11.72939,-64.88394,2022-01-01,18,GUAJARA-MIRIM
2,2022-01-01 18:13:00,AQUA_M-T,Brasil,ACRE,XAPURI,Amazônia,7.0,0.0,0.0,5.5,-10.97156,-68.33749,2022-01-01,18,XAPURI
3,2022-01-01 18:15:00,AQUA_M-T,Brasil,RONDÔNIA,NOVA MAMORÉ,Amazônia,3.0,0.0,0.1,23.2,-10.14938,-65.11926,2022-01-01,18,NOVA MAMORE
4,2022-01-01 18:15:00,AQUA_M-T,Brasil,AMAZONAS,BOCA DO ACRE,Amazônia,4.0,0.4,0.0,6.7,-9.05768,-67.50098,2022-01-01,18,BOCA DO ACRE


In [ ]:
df_queimadas.shape

(354018, 15)

In [ ]:
df_queimadas.columns.tolist()

['DataHora',
 'Satelite',
 'Pais',
 'Estado',
 'Municipio',
 'Bioma',
 'DiaSemChuva',
 'Precipitacao',
 'RiscoFogo',
 'FRP',
 'Latitude',
 'Longitude',
 'fecha',
 'hora',
 'municipio_norm']

In [ ]:
columnas_finales = [
    "fecha", "hora", "municipio_norm", "Estado",
    "DiaSemChuva", "Precipitacao", "RiscoFogo", "FRP",
    "Latitude", "Longitude"
]

df_queimadas_limpio = df_queimadas[columnas_finales].copy()

df_queimadas_limpio.to_csv("bdqueimadas_limpio.csv", index=False, encoding="utf-8")
print("bdqueimadas_limpio.csv guardado:", df_queimadas_limpio.shape)
print(df_queimadas_limpio.dtypes)

bdqueimadas_limpio.csv guardado: (354018, 10)
fecha             datetime64[ns]
hora                       int32
municipio_norm            object
Estado                    object
DiaSemChuva              float64
Precipitacao             float64
RiscoFogo                float64
FRP                      float64
Latitude                 float64
Longitude                float64
dtype: object


In [ ]:
print(df_deter[["classname", "dentro_area_protegida", "area_ha", "score_riesgo", "Nivel_Riesgo_Amenaza"]].sample(5))
print(df_deter["score_riesgo"].value_counts().sort_index())

                  classname  dentro_area_protegida  area_ha  score_riesgo  \
26611  CICATRIZ_DE_QUEIMADA                      0   202.85             4   
20953  CICATRIZ_DE_QUEIMADA                      0    16.65             2   
13814       DESMATAMENTO_CR                      0     9.97             3   
33018       DESMATAMENTO_CR                      0     8.12             3   
20921       DESMATAMENTO_CR                      0     9.56             3   

      Nivel_Riesgo_Amenaza  
26611             Moderado  
20953                 Bajo  
13814             Moderado  
33018             Moderado  
20921             Moderado  
score_riesgo
1     2313
2     4170
3    15342
4    17098
5     7743
6     3051
7      199
8       55
Name: count, dtype: int64
